# NaviTrace LLaVA‑v1.6 Fine-Tuning Notebook

This notebook finetunes the **LLaVA‑v1.6** model on the NaviTrace dataset and compares several parameter-efficient fine-tuning strategies:

- **BitFit** (bias-only updates)
- **IA3** as the **adapter-style** baseline
- **LoRA**
- **QLoRA**

This notebook uses the following high-level steps to fine-tune the model to the NaviTrace dataset:

1. Load and filter the NaviTrace data
2. Preprocess images and traces
3. Convert each sample into a multimodal instruction-tuning example
4. Fine-tune LLaVA‑v1.6 with multiple PEFT methods
5. Generate validation/test predictions
6. Evaluate with the same navigation metrics used in the original notebook
7. Compare method results in a summary table

Later, this notebook allows you to load fine-tuned CuDA checkpoints to allow for inferences of:
- in-house test dataset
- NaviTrace validation set
- NaviTrace test set

## References used for the notebook design

For the multimodal fine-tuning path, this notebook follows Hugging Face guidance that:
- LLaVA‑NeXT / LLaVA‑v1.6 is supported in Transformers and can be loaded with an `AutoProcessor` and image-text generation model class.
- TRL supports **vision-language supervised fine-tuning** with `SFTTrainer`, custom image-text collation, and optional PEFT integration.
- PEFT officially supports adapter-style and LoRA-style methods such as **IA3** and **LoRA**. 
- QLoRA uses a frozen **4-bit quantized** base model together with LoRA adapters, with `nf4` recommended for 4-bit training.
- Classic bottleneck adapters are documented by AdapterHub.

Need to install these:

In [ ]:
# !pip install -U datasets huggingface_hub torch torchvision Pillow numpy pandas scipy scikit-image tqdm matplotlib
# !pip install -U transformers accelerate trl peft bitsandbytes tensorboard sentencepiece
# !pip install -U bitsandbytes>=0.46.1

# Optional:
# !pip install -U hf_xet

Import necessary libraries:

In [ ]:
# Main environment setup

import os
import json
import io
import re
import ast
import json
import math
import time
import copy
import random
import pathlib
import shutil
import functools
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm
import io
import base64

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset, DatasetDict, load_dataset
from scipy.spatial import KDTree
from skimage.draw import line_aa
from skimage.draw import line as sk_line

import gc
from pathlib import Path

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
from peft import (
    LoraConfig,
    IA3Config,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"torch        : {torch.__version__}")
print(f"device       : {DEVICE}")
print(f"bf16 support : {torch.cuda.is_available() and torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")


In [ ]:
# Global configuration
# Comment: these settings mirror the structure of the original notebook where possible.

CFG = {
    # Data / task settings
    "embodiment": "Legged Robot",
    "val_split": 0.20,
    "seed": 42,

    # Quick experiment testing:
    "max_train_samples": None,      
    "max_val_samples": None,
    "max_test_samples": 500,

    # Model settings:
    "model_id": "llava-hf/llava-v1.6-mistral-7b-hf",
    "image_max_side": 672,           
    "max_new_tokens": 512,
    "generation_num_beams": 1,

    # Training settings
    "num_train_epochs": 200,
    "learning_rate": 1e-5, #2e-5
    "patience": 5, # Stop after this many eval epochs without validation-loss improvement
    "early_stopping_threshold": 0.0,
    "weight_decay": 1e-4,
    "warmup_ratio": 0.03,
    "per_device_train_batch_size": 1,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_total_limit": 1,
    "gradient_checkpointing": True,
    "report_to": "tensorboard", 

    # Formatting / output settings
    "trace_points": None,            
    "output_root": "./llava_navitrace_runs",
    "cache_root": "./llava_navitrace_cache",
    "penalty_tsv": "./category_penalty.tsv",
    "m2f_config": "./mask2former_config.json",
    "bad_score_threshold": 3234.75,
    "penalty_dist_thr": 35,

    # Method-specific default ranks
    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
}

os.makedirs(CFG["output_root"], exist_ok=True)
os.makedirs(CFG["cache_root"], exist_ok=True)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

CFG


## Data setup

- load `leggedrobotics/navitrace`
- keep only samples with valid ground-truth traces for the requested embodiment
- compute a fixed train/validation split
- compute the median number of trace waypoints so all traces can be resampled to a common length


In [ ]:
# Load NaviTrace dataset

ALL_SAMPLES_PKL = os.path.join(CFG["cache_root"], "samples_all.pkl")

if os.path.exists(ALL_SAMPLES_PKL):
    print("Loading cached filtered samples...")
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = torch.load(f, weights_only=False) if ALL_SAMPLES_PKL.endswith(".pt") else None

if os.path.exists(ALL_SAMPLES_PKL) and saved is None:
    import pickle
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = pickle.load(f)

if os.path.exists(ALL_SAMPLES_PKL):
    val_split_samples = saved["val_split"]
    print(f"NaviTrace filtered validation split : {len(val_split_samples)} samples")
else:
    print("Loading NaviTrace from Hugging Face...")
    dataset = load_dataset("leggedrobotics/navitrace")
    print(f"Available splits: {list(dataset.keys())}")

    val_split_samples = []
    skipped = 0
    for s in tqdm(list(dataset["validation"]), desc="Filtering validation"):
        gt = s["ground_truth"].get(CFG["embodiment"])
        if gt is not None and len(gt) > 0:
            val_split_samples.append(s)
        else:
            skipped += 1

    print(f"Kept={len(val_split_samples)}  Skipped={skipped}")

    import pickle
    with open(ALL_SAMPLES_PKL, "wb") as f:
        pickle.dump({"val_split": val_split_samples}, f)

# Compute a common number of waypoints from the median GT trace length
trace_lengths = []
for s in val_split_samples:
    for t in s["ground_truth"][CFG["embodiment"]]:
        trace_lengths.append(len(t))

CFG["trace_points"] = int(np.median(trace_lengths))
print(f"Common trace length N = {CFG['trace_points']}")

# Fixed split for reproducibility
random.seed(CFG["seed"])
random.shuffle(val_split_samples)

n_our_val = int(len(val_split_samples) * CFG["val_split"])
val_samples = val_split_samples[:n_our_val]
train_samples = val_split_samples[n_our_val:]

if CFG["max_train_samples"] is not None:
    train_samples = train_samples[:CFG["max_train_samples"]]
if CFG["max_val_samples"] is not None:
    val_samples = val_samples[:CFG["max_val_samples"]]

print(f"Train samples: {len(train_samples)}")
print(f"Val samples  : {len(val_samples)}")

## Data Preprocessing 

- pad images to square
- map traces into padded normalized coordinates
- resample traces to a fixed number of waypoints
- convert normalized trace predictions back to pixel coordinates


In [ ]:
def pad_to_square(image: Image.Image):
    w, h = image.size
    max_s = max(w, h)
    padded = Image.new("RGB", (max_s, max_s), (0, 0, 0))
    x_off = (max_s - w) // 2
    y_off = (max_s - h) // 2
    padded.paste(image, (x_off, y_off))
    return padded, (x_off / max_s, y_off / max_s, w / max_s, h / max_s)

def resize_if_needed(image: Image.Image, max_side: int):
    w, h = image.size
    longest = max(w, h)
    if longest <= max_side:
        return image
    scale = max_side / longest
    return image.resize((int(round(w * scale)), int(round(h * scale))), Image.BICUBIC)

def adjust_trace(trace_px, orig_w, orig_h, xof, yof, sx, sy):
    t = np.array(trace_px, dtype=float)
    if t.ndim != 2 or t.shape[1] != 2:
        raise ValueError("Trace must have shape (num_points, 2)")
    t[:, 0] = (t[:, 0] / orig_w) * sx + xof
    t[:, 1] = (t[:, 1] / orig_h) * sy + yof
    return t.astype(np.float32)

def interpolate_trace(trace: np.ndarray, N: int) -> np.ndarray:
    if len(trace) == 0:
        return np.zeros((N, 2), dtype=np.float32)
    if len(trace) == 1:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    diffs = np.diff(trace, axis=0)
    dists = np.sqrt((diffs ** 2).sum(axis=1))
    cumlen = np.concatenate([[0], np.cumsum(dists)])
    total = cumlen[-1]

    if total == 0:
        return np.tile(trace[0], (N, 1)).astype(np.float32)

    t_old = cumlen / total
    t_new = np.linspace(0, 1, N)
    out = np.stack(
        [
            np.interp(t_new, t_old, trace[:, 0]),
            np.interp(t_new, t_old, trace[:, 1]),
        ],
        axis=1,
    )
    return out.astype(np.float32)

def pred_padded_to_pixel(pred_norm, orig_w, orig_h):
    pred_norm = np.array(pred_norm, dtype=float)
    max_s = max(orig_w, orig_h)
    x_off = (max_s - orig_w) / 2.0
    y_off = (max_s - orig_h) / 2.0

    px = pred_norm.copy()
    px[:, 0] = px[:, 0] * max_s - x_off
    px[:, 1] = px[:, 1] * max_s - y_off

    px[:, 0] = np.clip(px[:, 0], 0, orig_w - 1)
    px[:, 1] = np.clip(px[:, 1], 0, orig_h - 1)
    return px.astype(np.float32)

def choose_canonical_trace(sample, N):
    # Main step comment: for generative supervision we need a single target sequence.
    gt_root = sample.get("ground_truth", None)
    if not isinstance(gt_root, dict):
        raise ValueError("Sample has no usable ground_truth dictionary")

    gt_traces = gt_root.get(CFG["embodiment"])
    if gt_traces is None or len(gt_traces) == 0:
        raise ValueError("Sample has no GT trace for the requested embodiment")

    gt_px = np.array(gt_traces[0], dtype=float)
    img = sample["image"]
    orig_w, orig_h = img.size
    _, (xof, yof, sx, sy) = pad_to_square(img)
    gt_norm = adjust_trace(gt_px, orig_w, orig_h, xof, yof, sx, sy)
    return interpolate_trace(gt_norm, N)

def compact_trace_json(trace_norm: np.ndarray, decimals: int = 4) -> str:
    # Main step comment: serialize the target trace as compact JSON for supervised generation.
    rounded = [[round(float(x), decimals), round(float(y), decimals)] for x, y in trace_norm]
    payload = {
        "goal": rounded[-1],
        "trace": rounded,
    }
    return json.dumps(payload, separators=(",", ":"))

def extract_first_json_object(text: str):
    # Main step comment: recover the model output as JSON even when generation contains extra text.
    if text is None:
        return None
    start = text.find("{")
    if start < 0:
        return None
    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                snippet = text[start:i + 1]
                try:
                    return json.loads(snippet)
                except Exception:
                    return None
    return None

def json_to_trace_norm(obj, N):
    # Main step comment: safely convert model JSON back to a fixed-length normalized trace.

    if not isinstance(obj, dict) or "trace" not in obj:
        return None

    raw_trace = obj["trace"]
    if not isinstance(raw_trace, (list, tuple)):
        return None

    points = []

    for item in raw_trace:
        # Standard [x, y]
        if isinstance(item, (list, tuple)) and len(item) == 2:
            a, b = item[0], item[1]

            # [x, y]
            if np.isscalar(a) and np.isscalar(b):
                try:
                    points.append([float(a), float(b)])
                    continue
                except (TypeError, ValueError):
                    pass

            # [[x, y], extra] or [extra, [x, y]] -> salvage the pair if present
            for candidate in (a, b):
                if isinstance(candidate, (list, tuple)) and len(candidate) == 2:
                    try:
                        points.append([float(candidate[0]), float(candidate[1])])
                        break
                    except (TypeError, ValueError):
                        pass
            else:
                pass
            continue

        # Nested singleton like [[x, y]]
        if (
            isinstance(item, (list, tuple))
            and len(item) == 1
            and isinstance(item[0], (list, tuple))
            and len(item[0]) == 2
        ):
            try:
                points.append([float(item[0][0]), float(item[0][1])])
                continue
            except (TypeError, ValueError):
                pass

        # Dict point {"x": ..., "y": ...}
        if isinstance(item, dict) and "x" in item and "y" in item:
            try:
                points.append([float(item["x"]), float(item["y"])])
                continue
            except (TypeError, ValueError):
                pass

        # Flat 2-number dict variants
        if isinstance(item, dict) and "point" in item:
            p = item["point"]
            if isinstance(p, (list, tuple)) and len(p) == 2:
                try:
                    points.append([float(p[0]), float(p[1])])
                    continue
                except (TypeError, ValueError):
                    pass

    if len(points) == 0:
        return None

    trace = np.asarray(points, dtype=np.float32)

    if trace.ndim != 2 or trace.shape[1] != 2:
        return None

    trace = np.clip(trace, 0.0, 1.0)

    if len(trace) == N:
        return trace.tolist()

    if len(trace) == 1:
        return np.repeat(trace, N, axis=0).tolist()

    old_idx = np.linspace(0.0, 1.0, len(trace))
    new_idx = np.linspace(0.0, 1.0, N)
    x_new = np.interp(new_idx, old_idx, trace[:, 0])
    y_new = np.interp(new_idx, old_idx, trace[:, 1])

    return np.stack([x_new, y_new], axis=1).tolist()


In [ ]:
# import inspect
# print(inspect.getsource(json_to_trace_norm))

## Build multimodal instruction-tuning examples
  
For LLaVA, each sample is turned into a **conversation**:

- **system**: explain the output format
- **user**: provide the image and task text
- **assistant**: return the target trajectory JSON

This lets us fine-tune LLaVA as a vision-language sequence model while still preserving the navigation objective.


In [ ]:
# SYSTEM_PROMPT = (
#     "You are a visual navigation model for the NaviTrace benchmark. "
#     "Given a scene image and a navigation instruction, predict a feasible 2D path for the "
#     "Legged Robot as normalized waypoints in the padded-square image frame. "
#     "Return JSON only with keys 'goal' and 'trace'. "
#     "'goal' must be the last waypoint [x, y]. "
#     "'trace' must be an ordered list of [x, y] pairs with values in [0, 1]. "
#     "The starting point should be at the location of [0.5*x, 0.95*y] of the original input image size. "
#     "Do not add explanations."
# )

SYSTEM_PROMPT = (
"""You are a navigation expert for various embodiments including robots and humans. Given an image of the current scenario, a legged robot embodiment, and a navigation task (e.g., "Go down the road"), you will predict a feasible future trajectory as a sequence of 2D points in normalized image coordinates (ranging from 0 to 1, where [0,0] is the top-left and [1,1] is the bottom-right).

- The image shows a first-person view of the navigation scenario
- Start your trajectory near the bottom center of the image, which corresponds approximately to normalized coordinate [0.5, 0.95] (representing the current position of the embodiment)
- The trajectory should be adapted to the embodiment's abilities and limitations
- Plan the path forward from this starting position based on what the embodiment can see and navigate
- The trajectory should extend all the way to the goal if the path is visible. If the path is occluded, the trajectory should end where the path becomes fully obscured, unless the path can be reasonably inferred from the visible context.
- If a red traffic light is visible and affects the planned path, or if crossing traffic or moving vehicles are present that make it unsafe to proceed, stop at an appropriate waiting position (e.g., just before the intersection or curb) and end the trajectory there.
- All tasks that you are given have a solution
- Output **only** the list of 2D points in normalized image coordinates (values between 0 and 1) in the following format: `[[x1, y1], [x2, y2], ..., [xn, yn]]`
- Do not include any explanation or additional output

### Embodiment Movement Characteristics

- **Legged Robot**: A quadruped like ANYmal. Behaves similarly to a human, but it is shorter. It can handle stairs and escalators.
"""    
)

def get_sample_id(sample):
    return (
        sample.get("sample_id")
        or sample.get("id")
        or sample.get("uid")
        or "unknown"
    )

def get_sample_task(sample):
    for key in ["task", "instruction", "text", "prompt", "command", "query", "description"]:
        val = sample.get(key, None)
        if isinstance(val, str) and val.strip():
            return val
    raise KeyError(f"No task/instruction field found. Keys: {list(sample.keys())}")

def get_sample_category(sample):
    return sample.get("category", sample.get("scene_category", "unknown"))

def make_messages_for_sample(sample, include_answer=True):
    # Main step comment: package each sample as a multimodal chat conversation.
    img = sample["image"].convert("RGB")
    img = resize_if_needed(img, CFG["image_max_side"])

    task_text = get_sample_task(sample)

    user_text = (
        f"Task: {task_text}\n"
        f"Embodiment: {CFG['embodiment']}\n"
        f"Predict the path as JSON only."
    )

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": user_text},
            ],
        },
    ]

    if include_answer:
        trace_norm = choose_canonical_trace(sample, CFG["trace_points"])
        answer = compact_trace_json(trace_norm)
        messages.append(
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        )

    return messages


def build_sft_dataset(samples, include_answer=True):
    rows = []
    for s in tqdm(samples, desc="Formatting messages"):
        try:
            rows.append(
                {
                    "sample_id": s["sample_id"],
                    "task": s["task"],
                    "category": s["category"],
                    "messages": make_messages_for_sample(s, include_answer=include_answer),
                }
            )
        except Exception as e:
            print(f"Skipping sample {s.get('sample_id', 'unknown')} due to formatting error: {e}")
    return Dataset.from_list(rows)


print("train sample keys:", train_samples[0].keys())
print("val sample keys:", val_samples[0].keys())

print(make_messages_for_sample(train_samples[0], include_answer=True))

train_ds = build_sft_dataset(train_samples, include_answer=True)
val_ds = build_sft_dataset(val_samples, include_answer=True)

train_ds, val_ds


In [ ]:
# Inspect one formatted training example

example = train_ds[0]
example["messages"]

Load processor:


In [ ]:
processor = AutoProcessor.from_pretrained(CFG["model_id"])
processor.tokenizer.padding_side = "right"

# Some checkpoints may require these fields to be present on the processor for image-token expansion.
# They are mentioned in HF discussions for LLaVA-NeXT processors.
if not hasattr(processor, "patch_size") and hasattr(processor, "image_processor"):
    processor.patch_size = getattr(processor.image_processor, "patch_size", None)

if not hasattr(processor, "vision_feature_select_strategy"):
    processor.vision_feature_select_strategy = "default"

print(type(processor))
print(processor.tokenizer.__class__.__name__)


Training utilities

In [ ]:
METHODS = {
    "bitfit": {
        "description": "Bias-only fine-tuning (BitFit)",
    },
    "ia3_adapter": {
        "description": "Adapter-style IA3 via PEFT",
    },
    "lora": {
        "description": "Standard LoRA on all linear layers",
    },
    "qlora": {
        "description": "4-bit quantized base model + LoRA (QLoRA)",
    },
}

METHODS


In [ ]:
def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100.0 * trainable / total if total > 0 else 0.0
    return {"trainable": trainable, "total": total, "pct": pct}

def print_trainable_summary(model, title="Model"):
    stats = count_trainable_parameters(model)
    print(
        f"{title}: trainable={stats['trainable']:,} / total={stats['total']:,} "
        f"({stats['pct']:.4f}%)"
    )

# BitFit trains only bias parameters.
def set_bitfit_trainable(model):
    for p in model.parameters():
        p.requires_grad = False

    for name, p in model.named_parameters():
        if name.endswith(".bias") or name == "bias":
            p.requires_grad = True

    return model

# Load the correct base model depending on whether quantization is needed.
def load_base_model_for_method(method_name: str):
    common_kwargs = {
        "torch_dtype": DTYPE,
        "low_cpu_mem_usage": True,
        "device_map": "auto" if torch.cuda.is_available() else None,
    }

    if method_name == "qlora":
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            bnb_4bit_quant_storage=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        )
        model = AutoModelForImageTextToText.from_pretrained(CFG["model_id"], quantization_config=quant_config, **common_kwargs)
        model = prepare_model_for_kbit_training(model)
        return model

    model = AutoModelForImageTextToText.from_pretrained(CFG["model_id"], **common_kwargs)
    return model

# Wrap the model according to the chosen PEFT strategy.
def apply_peft_or_bitfit(model, method_name: str):
    if method_name == "bitfit":
        model = set_bitfit_trainable(model)
        return model

    if method_name == "ia3_adapter":
        peft_cfg = IA3Config(
            task_type="CAUSAL_LM",
            target_modules=["k_proj", "v_proj", "down_proj"],
            feedforward_modules=["down_proj"],
        )
        model = get_peft_model(model, peft_cfg)
        return model

    if method_name in ("lora", "qlora"):
        peft_cfg = LoraConfig(
            r=CFG["lora_r"],
            lora_alpha=CFG["lora_alpha"],
            lora_dropout=CFG["lora_dropout"],
            bias="none",
            target_modules="all-linear",
            task_type="CAUSAL_LM",
            modules_to_save=["lm_head", "embed_tokens"],
        )
        model = get_peft_model(model, peft_cfg)
        return model

    raise ValueError(f"Unknown method: {method_name}")


Safe Image Loading Methods:

In [ ]:
def to_pil_image_safe(obj):
    if obj is None:
        return None

    if isinstance(obj, Image.Image):
        return obj.convert("RGB")

    if isinstance(obj, dict):
        if obj.get("bytes") is not None:
            return Image.open(io.BytesIO(obj["bytes"])).convert("RGB")
        if obj.get("path"):
            return Image.open(obj["path"]).convert("RGB")
        if obj.get("base64") is not None:
            return Image.open(io.BytesIO(base64.b64decode(obj["base64"]))).convert("RGB")

    raise TypeError(f"Unsupported image object type: {type(obj)}")


def gather_images_from_messages(messages):
    images = []
    for message in messages:
        content = message.get("content", [])
        if not isinstance(content, list):
            continue

        for item in content:
            if not isinstance(item, dict):
                continue

            if item.get("type") == "image" and item.get("image") is not None:
                pil_img = to_pil_image_safe(item["image"])
                if pil_img is not None:
                    images.append(pil_img)

    return images

def build_collate_fn(processor, train_on_assistant_only=False):
    
    # create a VLM batch that includes both the chat text and the image tensors.
    def collate_fn(examples):
        texts = [
            processor.apply_chat_template(
                ex["messages"],
                tokenize=False,
                add_generation_prompt=False,
            )
            for ex in examples
        ]

        # Convert serialized images back to PIL and flatten single-image examples
        image_lists = [gather_images_from_messages(ex["messages"]) for ex in examples]

        images = []
        for img_list in image_lists:
            if len(img_list) == 0:
                raise ValueError("No image found in example messages")
            elif len(img_list) == 1:
                images.append(img_list[0])   # flat single-image case
            else:
                images.append(img_list)      # keep multi-image case if ever used

        batch = processor(
            text=texts,
            images=images,
            padding=True,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()

        if processor.tokenizer.pad_token_id is not None:
            labels[labels == processor.tokenizer.pad_token_id] = -100

        image_token_candidates = set()
        for tok_name in ["image_token", "boi_token", "eoi_token"]:
            tok = processor.tokenizer.special_tokens_map.get(tok_name, None)
            if tok is not None:
                try:
                    image_token_candidates.add(processor.tokenizer.convert_tokens_to_ids(tok))
                except Exception:
                    pass

        image_token_candidates.add(262144)

        for tok_id in image_token_candidates:
            labels[labels == tok_id] = -100

        if train_on_assistant_only:
            pass

        batch["labels"] = labels
        return batch

    return collate_fn


Model Helper fundtions: load the model, apply the chosen tuning strategy, train, and save logs.

In [ ]:
class HistoryCallback(TrainerCallback):
    def __init__(self):
        self.history = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            row = {"step": state.global_step, **logs}
            self.history.append(row)


def infer_batch_size(inputs):
    if isinstance(inputs, torch.Tensor):
        return inputs.shape[0] if inputs.ndim > 0 else 1
    if isinstance(inputs, dict):
        for value in inputs.values():
            size = infer_batch_size(value)
            if size is not None:
                return size
    if isinstance(inputs, (list, tuple)):
        for value in inputs:
            size = infer_batch_size(value)
            if size is not None:
                return size
    return None

def compute_initial_train_loss(trainer):
    dataloader = trainer.get_train_dataloader()
    model = trainer.model
    was_training = model.training
    model.eval()

    total_loss = 0.0
    total_items = 0
    start_time = time.time()

    for inputs in dataloader:
        batch_size = infer_batch_size(inputs) or 1
        inputs = trainer._prepare_inputs(inputs)

        with torch.no_grad():
            try:
                loss = trainer.compute_loss(model, inputs, return_outputs=False)
            except TypeError:
                loss = trainer.compute_loss(model, inputs)

        if isinstance(loss, (tuple, list)):
            loss = loss[0]

        total_loss += float(loss.detach().cpu()) * batch_size
        total_items += batch_size

    if was_training:
        model.train()

    train_loss = total_loss / max(total_items, 1)
    runtime = time.time() - start_time
    return {
        "loss": train_loss,
        "train_loss_runtime": runtime,
        "train_loss_samples_per_second": total_items / runtime if runtime > 0 else None,
        "train_loss_steps_per_second": len(dataloader) / runtime if runtime > 0 else None,
    }


def compute_initial_eval_loss(trainer):
    dataloader = trainer.get_eval_dataloader()
    model = trainer.model
    was_training = model.training
    model.eval()

    total_loss = 0.0
    total_items = 0
    start_time = time.time()

    for inputs in dataloader:
        batch_size = infer_batch_size(inputs) or 1
        inputs = trainer._prepare_inputs(inputs)

        with torch.no_grad():
            try:
                loss = trainer.compute_loss(model, inputs, return_outputs=False)
            except TypeError:
                loss = trainer.compute_loss(model, inputs)

        if isinstance(loss, (tuple, list)):
            loss = loss[0]
        total_loss += float(loss.detach().cpu()) * batch_size
        total_items += batch_size

    if was_training:
        model.train()

    eval_loss = total_loss / max(total_items, 1)
    runtime = time.time() - start_time
    return {
        "eval_loss": eval_loss,
        "eval_runtime": runtime,
        "eval_samples_per_second": total_items / runtime if runtime > 0 else None,
        "eval_steps_per_second": len(dataloader) / runtime if runtime > 0 else None,
    }


def make_training_args(method_name: str, output_dir: str):
    bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    fp16_ok = torch.cuda.is_available() and not bf16_ok

    return TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=CFG["num_train_epochs"],
        learning_rate=CFG["learning_rate"],
        weight_decay=CFG["weight_decay"],
        warmup_ratio=CFG["warmup_ratio"],
        per_device_train_batch_size=CFG["per_device_train_batch_size"],
        per_device_eval_batch_size=CFG["per_device_eval_batch_size"],
        gradient_accumulation_steps=CFG["gradient_accumulation_steps"],
        logging_steps=CFG["logging_steps"],
        save_strategy=CFG["save_strategy"],
        eval_strategy=CFG["eval_strategy"],
        save_total_limit=CFG["save_total_limit"],
        gradient_checkpointing=CFG["gradient_checkpointing"],
        remove_unused_columns=False,   # important for multimodal examples
        report_to=CFG["report_to"],
        bf16=bf16_ok,
        fp16=fp16_ok,
        dataloader_num_workers=0,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )


def train_one_method(method_name: str, train_ds: Dataset, val_ds: Dataset):
    run_dir = os.path.join(CFG["output_root"], method_name)
    os.makedirs(run_dir, exist_ok=True)

    model = load_base_model_for_method(method_name)
    model = apply_peft_or_bitfit(model, method_name)

    if hasattr(model, "config"):
        model.config.use_cache = False

    print_trainable_summary(model, title=f"{method_name} model")

    collate_fn = build_collate_fn(processor)
    training_args = make_training_args(method_name, run_dir)
    history_cb = HistoryCallback()

    early_stopping_cb = EarlyStoppingCallback(
        early_stopping_patience=CFG["patience"],
        early_stopping_threshold=CFG["early_stopping_threshold"],
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collate_fn,
        callbacks=[history_cb, early_stopping_cb],
    )

    initial_train_metrics = compute_initial_train_loss(trainer)
    initial_eval_metrics = compute_initial_eval_loss(trainer)

    history_cb.history.append({
        "step": 0,
        "epoch": 0.0,
        **initial_train_metrics,
        **initial_eval_metrics,
    })


    start = time.time()
    trainer.train()
    train_time_sec = time.time() - start

    # Save best model in run_dir so checkpoint map can load it later.
    saved_model_dir = run_dir
    trainer.save_model(saved_model_dir)
    processor.save_pretrained(saved_model_dir)

    parameter_file = None
    for candidate_name in ["adapter_model.safetensors", "model.safetensors", "pytorch_model.bin"]:
        candidate_path = os.path.join(saved_model_dir, candidate_name)
        if os.path.exists(candidate_path):
            parameter_file = candidate_path
            break

    saved_model_manifest = {
        "method": method_name,
        "saved_model_dir": saved_model_dir,
        "parameter_file": parameter_file,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
        "load_with_checkpoint_map_value": saved_model_dir,
    }
    manifest_path = os.path.join(run_dir, "saved_model_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(saved_model_manifest, f, indent=2)

    print(f"Saved inference-ready model directory to: {saved_model_dir}")
    print(f"Saved parameter file to: {parameter_file}")
    print(f"Saved model manifest to: {manifest_path}")

    hist_df = pd.DataFrame(history_cb.history)
    hist_path = os.path.join(run_dir, "history.csv")
    hist_df.to_csv(hist_path, index=False)

    return {
        "method": method_name,
        "run_dir": run_dir,
        "trainer": trainer,
        "model": trainer.model,
        "history": hist_df,
        "train_time_sec": train_time_sec,
        "num_train_epochs": CFG["num_train_epochs"],
        "saved_model_dir": saved_model_dir,
        "parameter_file": parameter_file,
        "manifest_path": manifest_path,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "best_eval_loss": trainer.state.best_metric,
    }


## Trace evaluation helpers

Helps with Trace evaluation by calculating trace scores

- semantic penalty
- FDE
- DTW
- normalized trace score

Predictions are generated by LLaVA as JSON and then parsed back into waypoint arrays.

In [ ]:
@functools.lru_cache(maxsize=4)
def penalty_lookup(embodiment):
    df = pd.read_csv(CFG["penalty_tsv"], sep="\t")
    with open(CFG["m2f_config"]) as f:
        m2f = json.load(f)
    id2label = {int(k): v for k, v in m2f["id2label"].items()}
    lkp = {}
    for lid, name in id2label.items():
        row = df[df["category"] == name]
        lkp[lid] = float(row.iloc[0][embodiment]) * 0.8 if len(row) else 0.0
    return lkp

def rasterize(trace, H, W):
    pts, pixels = np.array(trace), []
    if len(pts) > 1:
        for i in range(len(pts) - 1):
            r0, c0 = int(round(pts[i][1])), int(round(pts[i][0]))
            r1, c1 = int(round(pts[i + 1][1])), int(round(pts[i + 1][0]))
            rr, cc, _ = line_aa(r0, c0, r1, c1)
            v = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
            pixels.extend(zip(rr[v], cc[v]))
    elif len(pts) == 1:
        r, c = int(round(pts[0][1])), int(round(pts[0][0]))
        if 0 <= r < H and 0 <= c < W:
            pixels.append((r, c))
    return np.array(pixels)

def penalty_mask(seg, gt_px, emb, dthr=35):
    H, W = seg.shape
    mask = np.zeros((H, W), dtype=float)
    lkp = penalty_lookup(emb)
    gtp = rasterize(gt_px, H, W)
    if len(gtp) == 0:
        return mask

    tree = KDTree(gtp)
    ids = seg.ravel()
    und = np.where(np.isin(ids, list(lkp.keys())))[0]
    if und.size == 0:
        return mask

    rows, cols = np.unravel_index(und, (H, W))
    coords = np.vstack((rows, cols)).T
    dist, _ = tree.query(coords)
    pen_idx = coords[dist > dthr]

    if pen_idx.size > 0:
        rp, cp = pen_idx[:, 0], pen_idx[:, 1]
        mask[rp, cp] = np.vectorize(lkp.get)(seg[rp, cp], 0)

    return mask

def sem_penalty(pred_px, pmask):
    H, W = pmask.shape
    vals = []
    for i in range(len(pred_px) - 1):
        x1, y1 = int(round(pred_px[i][0])), int(round(pred_px[i][1]))
        x2, y2 = int(round(pred_px[i + 1][0])), int(round(pred_px[i + 1][1]))
        rr, cc = sk_line(y1, x1, y2, x2)
        v = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
        vals.extend(pmask[rr[v], cc[v]].tolist())
    return float(np.mean(vals)) if vals else 0.0

def calc_fde(p, g):
    return float(np.linalg.norm(np.array(p[-1]) - np.array(g[-1])))

def calc_dtw(p, g):
    p = np.array(p, dtype=float)
    g = np.array(g, dtype=float)

    if len(p) != len(g):
        longer, shorter = (p, g) if len(p) >= len(g) else (g, p)
        d = np.cumsum([0] + [np.linalg.norm(shorter[i] - shorter[i - 1]) for i in range(1, len(shorter))])
        tot = d[-1]
        if tot > 0:
            d = d / tot
        t = np.linspace(0, 1, len(longer))
        shorter = np.column_stack(
            [
                np.interp(t, d, shorter[:, 0]),
                np.interp(t, d, shorter[:, 1]),
            ]
        )
        p, g = (longer, shorter) if len(p) >= len(g) else (shorter, longer)

    n, m = len(p), len(g)
    D = np.full((n + 1, m + 1), np.inf)
    D[0, 0] = 0
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = np.linalg.norm(p[i - 1] - g[j - 1]) + min(
                D[i - 1, j], D[i, j - 1], D[i - 1, j - 1]
            )
    return float(D[n, m])

def norm_score(raw):
    return (CFG["bad_score_threshold"] - raw) / CFG["bad_score_threshold"] * 100.0


In [ ]:
# Retrieve the text instruction from any supported dataset schema.
def get_sample_instruction(sample):
    candidate_keys = [
        "instruction",
        "text",
        "prompt",
        "command",
        "query",
        "description",
        "goal",
        "task",
    ]

    for k in candidate_keys:
        v = sample.get(k, None)
        if isinstance(v, str) and v.strip():
            return v

    raise KeyError(f"No instruction-like field found. Available keys: {list(sample.keys())}")

# Generate Trace in a json format for validation samples
@torch.no_grad()
def generate_trace_json_val(model, sample):
    prompt_messages = make_messages_for_sample(sample, include_answer=False)
    prompt_text = processor.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    images = gather_images_from_messages(prompt_messages)

    inputs = processor(
        text=[prompt_text],
        images=images,
        return_tensors="pt",
        padding=True,
    )

    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    generated = model.generate(
        **inputs,
        max_new_tokens=CFG["max_new_tokens"],   # try 512, 768, or 1024 in CFG
        num_beams=CFG["generation_num_beams"],
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    completion_ids = generated[:, prompt_len:]

    # print("Generated token count:", completion_ids.shape[1])
    # print("Configured max_new_tokens:", CFG["max_new_tokens"])

    text = processor.batch_decode(completion_ids, skip_special_tokens=True)[0]

    # print("Generated text length:", len(text))
    # print("Last 300 chars:")
    # print(text[-300:])

    # print(
    #     "Bracket counts:",
    #     {
    #         "{": text.count("{"),
    #         "}": text.count("}"),
    #         "[": text.count("["),
    #         "]": text.count("]"),
    #     },
    # )

    return text

# Generate Trace in a json format for test samples
@torch.no_grad()
def generate_trace_json_test(model, sample):

    image = sample["image"]
    instruction = get_sample_instruction(sample)

    prompt = (
        f"You are predicting a normalized navigation trace for a {CFG['embodiment']}.\n"
        f"Given the image and instruction, output JSON only in the format:\n"
        f'{{"trace": [[x1, y1], [x2, y2], ..., [{CFG["trace_points"]} points total]]}}\n'
        f"Instruction: {instruction}"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

    gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
    gen_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0]
    return gen_text

# Generate validation predictions and score them with the original navigation metrics.
def evaluate_method_on_val(model, samples):
    nt_scores = []
    dtw_scores = []
    fde_scores = []
    pen_scores = []
    invalid_json = 0
    raw_generations = []

    for sample in tqdm(samples, desc="Evaluating val"):
        gen_text = generate_trace_json_val(model, sample)
        raw_generations.append(
            {
                "sample_id": sample["sample_id"],
                "text": gen_text,
            }
        )

        payload = extract_first_json_object(gen_text)
        try:
            pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
        except Exception:
            print("\nBad payload:")
            print(payload)
            raise

        if pred_norm is None:
            invalid_json += 1
            # Fallback: use a degenerate center trace so the sample can still be scored
            pred_norm = np.tile(np.array([[0.5, 0.5]], dtype=np.float32), (CFG["trace_points"], 1))

        img = sample["image"]
        orig_w, orig_h = img.size
        seg = np.array(sample["segmentation_mask"])
        trace_px = pred_padded_to_pixel(pred_norm, orig_w, orig_h)

        gt_traces = sample["ground_truth"].get(CFG["embodiment"])
        best_raw = best_d = best_f = best_p = float("inf")

        for gt_px in gt_traces:
            pmask = penalty_mask(seg, gt_px, CFG["embodiment"], CFG["penalty_dist_thr"])
            p_val = sem_penalty(trace_px, pmask)
            f_val = calc_fde(trace_px, gt_px)
            d_val = calc_dtw(trace_px, gt_px)
            raw = d_val + f_val + p_val
            if raw < best_raw:
                best_raw, best_d, best_f, best_p = raw, d_val, f_val, p_val

        nt_scores.append(norm_score(best_raw))
        dtw_scores.append(best_d)
        fde_scores.append(best_f)
        pen_scores.append(best_p)

    metrics = {
        "trace_score_mean": float(np.mean(nt_scores)),
        "trace_score_std": float(np.std(nt_scores)),
        "dtw_mean": float(np.mean(dtw_scores)),
        "fde_mean": float(np.mean(fde_scores)),
        "penalty_mean": float(np.mean(pen_scores)),
        "invalid_json": int(invalid_json),
        "num_scored": int(len(nt_scores)),
    }

    return metrics, pd.DataFrame(raw_generations)

def nonempty_value(*values, default=""):
    for value in values:
        if value is None:
            continue
        if isinstance(value, float) and pd.isna(value):
            continue
        if isinstance(value, str) and value.strip() == "":
            continue
        return value
    return default


def load_test_metadata(metadata_tsv=None):
    if metadata_tsv is None:
        candidate = pathlib.Path("test_predictions_simple.tsv")
        if not candidate.exists():
            candidate = pathlib.Path("vln-project") / "notebooks" / "test_predictions_simple.tsv"
        metadata_tsv = candidate if candidate.exists() else None

    if metadata_tsv is None:
        return None

    metadata_path = pathlib.Path(metadata_tsv)
    candidates = [metadata_path]
    if not metadata_path.is_absolute():
        candidates.extend([
            pathlib.Path.cwd() / metadata_path,
            pathlib.Path.cwd() / "vln-project" / "notebooks" / metadata_path,
        ])

    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, sep="\t", dtype={"sample_id": str})

    return None


def format_category_for_tsv(category):
    if isinstance(category, (list, tuple)):
        return json.dumps(list(category))
    return str(category)


def sample_metadata_for_export(sample, index, metadata_df=None, embodiment=None):
    metadata_row = metadata_df.iloc[index] if metadata_df is not None and index < len(metadata_df) else {}
    row_get = metadata_row.get if hasattr(metadata_row, "get") else (lambda key, default=None: default)
    default_embodiment = embodiment or (CFG.get("embodiment", "Legged Robot") if "CFG" in globals() else "Legged Robot")

    sample_id = nonempty_value(
        sample.get("sample_id"),
        sample.get("id"),
        sample.get("uid"),
        row_get("sample_id", ""),
    )
    category = nonempty_value(
        sample.get("category"),
        sample.get("scene_category"),
        row_get("category", ""),
        default="unknown",
    )
    embodiment = nonempty_value(
        sample.get("embodiment"),
        row_get("embodiment", ""),
        default_embodiment,
    )

    return {
        "sample_id": str(sample_id),
        "embodiment": str(embodiment),
        "category": format_category_for_tsv(category),
    }


# Compute score for single pair of traces
def score_single_trace_pair(pred_trace, gt_trace, sample=None, embodiment=None):
    embodiment = embodiment or CFG["embodiment"]

    pred_norm = np.asarray(pred_trace, dtype=np.float32)
    gt_norm = np.asarray(gt_trace, dtype=np.float32)

    img = sample["image"] if sample is not None else None
    if img is None:
        raise ValueError("sample is required to convert normalized traces back to pixel space")

    orig_w, orig_h = img.size
    pred_px = pred_padded_to_pixel(pred_norm, orig_w, orig_h)
    gt_px = pred_padded_to_pixel(gt_norm, orig_w, orig_h)

    dtw_val = calc_dtw(pred_px, gt_px)
    fde_val = calc_fde(pred_px, gt_px)

    penalty_val = 0.0
    if sample is not None and "segmentation_mask" in sample:
        seg = np.array(sample["segmentation_mask"])
        pmask = penalty_mask(seg, gt_px, embodiment, CFG["penalty_dist_thr"])
        penalty_val = sem_penalty(pred_px, pmask)

    raw_score = dtw_val + fde_val + penalty_val

    return {
        "dtw": float(dtw_val),
        "fde": float(fde_val),
        "penalty": float(penalty_val),
        "raw_score": float(raw_score),
        "trace_score": float(norm_score(raw_score)),
    }

# Only compute geometric scores for trace pair
def score_trace_pair_geom_only(pred_trace, gt_trace, sample):
    pred_norm = np.asarray(pred_trace, dtype=np.float32)
    gt_norm = np.asarray(gt_trace, dtype=np.float32)

    orig_w, orig_h = sample["image"].size
    pred_px = pred_padded_to_pixel(pred_norm, orig_w, orig_h)
    gt_px = pred_padded_to_pixel(gt_norm, orig_w, orig_h)

    dtw_val = calc_dtw(pred_px, gt_px)
    fde_val = calc_fde(pred_px, gt_px)
    raw_score = dtw_val + fde_val

    return {
        "dtw": float(dtw_val),
        "fde": float(fde_val),
        "raw_score": float(raw_score),
        "trace_score": float(norm_score(raw_score)),
    }


In [ ]:
# # test w/ Single Batch:
# tmp_collate = build_collate_fn(processor)
# tmp_batch = tmp_collate([train_ds[0], train_ds[1]])
# for k, v in tmp_batch.items():
#     if hasattr(v, "shape"):
#         print(k, v.shape)
#     else:
#         print(k, type(v))

Test generated traces:

In [ ]:
# Run model training
# result = train_one_method("lora", train_ds, val_ds)

# Test the test sample
# sample = test_samples[0]
# print(sample.keys())
# print(sample.get("ground_truth", None))
# gen_text = generate_trace_json_test(result["model"], sample)
# print(gen_text)


def pad_image_to_square_for_display(img):
    w, h = img.size
    side = max(w, h)
    pad_left = (side - w) // 2
    pad_top = (side - h) // 2
    pad_right = side - w - pad_left
    pad_bottom = side - h - pad_top
    return ImageOps.expand(img, border=(pad_left, pad_top, pad_right, pad_bottom), fill=0)

def trace_test_output_dir(result):
    method_name = result.get("method", "unknown_method")
    epochs = result.get("num_train_epochs", CFG["num_train_epochs"])
    try:
        epochs_label = f"{int(epochs)}_epochs" if float(epochs).is_integer() else f"{epochs}_epochs"
    except Exception:
        epochs_label = f"{epochs}_epochs"

    out_dir = os.path.join(CFG["output_root"], method_name, epochs_label)
    os.makedirs(out_dir, exist_ok=True)
    return out_dir

def get_best_epoch_for_result(result):
    history = result.get("history")
    if history is None or len(history) == 0:
        hist_path = os.path.join(result.get("run_dir", ""), "history.csv")
        if os.path.exists(hist_path):
            history = pd.read_csv(hist_path)

    if history is None or len(history) == 0 or "eval_loss" not in history.columns:
        return None

    eval_hist = history[history["eval_loss"].notna()]
    if eval_hist.empty:
        return None

    best_idx = eval_hist["eval_loss"].idxmin()
    return float(eval_hist.loc[best_idx, "epoch"])

def save_trace_overlay(sample, gt_trace, pred_trace, out_path, title=None):
    img = sample["image"].convert("RGB")
    img_sq = pad_image_to_square_for_display(img)

    gt = np.array(gt_trace)
    pr = np.array(pred_trace)

    side = img_sq.size[0]
    gt_px = gt * side
    pr_px = pr * side

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(img_sq)
    # ax.plot(gt_px[:, 0], gt_px[:, 1], marker="o", linewidth=2, label="GT")
    # ax.plot(pr_px[:, 0], pr_px[:, 1], marker="x", linewidth=2, label="Pred")
    ax.plot(gt_px[:, 0], gt_px[:, 1], marker="o", linewidth=2.5, color="lime", label="GT path")
    ax.scatter(gt_px[0, 0], gt_px[0, 1], s=90, marker="s", color="lime", edgecolor="black", label="GT start")
    ax.scatter(gt_px[-1, 0], gt_px[-1, 1], s=130, marker="*", color="lime", edgecolor="black", label="GT goal")
    ax.plot(pr_px[:, 0], pr_px[:, 1], marker="o", linewidth=2.5, color="red", label="Predicted path")
    ax.scatter(pr_px[0, 0], pr_px[0, 1], s=90, marker="s", color="red", edgecolor="black", label="Predicted start")
    ax.scatter(pr_px[-1, 0], pr_px[-1, 1], s=130, marker="*", color="red", edgecolor="black", label="Predicted goal")
    ax.set_axis_off()
    ax.legend(loc="upper right")
    if title:
        ax.set_title(title)
    fig.tight_layout(pad=0)
    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

# Check validation samples and compare predicted Trace vs. GT trace and compute scores
def run_trace_test(result, num_samples):
    out_dir = trace_test_output_dir(result)
    best_epoch = get_best_epoch_for_result(result)
    score_rows = []

    for i in range(num_samples):
        sample_number = i + 1
        sample = val_samples[i]
        text_prompt = get_sample_task(sample)

        # GT used for scoring
        gt_trace = choose_canonical_trace(sample, CFG["trace_points"])

        # model output after parsing
        gen_text = generate_trace_json_val(result["model"], sample)
        payload = extract_first_json_object(gen_text)
        pred_trace = json_to_trace_norm(payload, CFG["trace_points"])

        if pred_trace is None:
            pred_trace = np.tile(np.array([[0.5, 0.5]], dtype=np.float32), (CFG["trace_points"], 1))

        # print("\n")
        # print(sample.keys())
        # print("sample number:", sample_number)
        # print("sample id:", sample.get("sample_id", sample.get("id")))
        # print("GT first 5:", gt_trace[:5] if gt_trace is not None else None)
        # print("Pred first 5:", pred_trace[:5] if pred_trace is not None else None)
        # print("Generated text:", gen_text)
        # print("Parsed payload:", payload)

        pred = np.array(pred_trace)
        # print("min:", pred.min(axis=0), "max:", pred.max(axis=0))
        # print("Generated length:", len(gen_text))
        # print("Last 300 chars:")
        # print(gen_text[-300:])
        # print("val spread", gt_trace.std(axis=0))
        # print("pred spread:", pred.std(axis=0))

        # Test the trace generation on the validation sample and normalized frame.
        gt = np.array(gt_trace)
        pr = np.array(pred_trace)

        # plt.figure(figsize=(6, 6))
        # plt.plot(gt[:, 0], gt[:, 1], marker="o", label="GT")
        # plt.plot(pr[:, 0], pr[:, 1], marker="x", label="Pred")
        # plt.xlim(0, 1)
        # plt.ylim(1, 0)
        # plt.legend()
        # plt.grid(True)
        # plt.show()

        metrics = score_single_trace_pair(pred_trace, gt_trace, sample=sample)
        geom_metrics = score_trace_pair_geom_only(pred_trace, gt_trace, sample=sample)
        semantic_score = metrics["trace_score"]
        geometric_score = geom_metrics["trace_score"]

        image_path = os.path.join(out_dir, f"sample_{sample_number:03d}_trace_overlay.png")
        save_trace_overlay(
            sample,
            gt_trace,
            pred_trace,
            image_path,
            title=f"Sample {sample_number}: GT vs Pred",
        )

        print("Score w/ semantics: ", semantic_score)
        print("Score w/ only geometry: ", geometric_score)
        print("Saved overlay:", image_path)

        score_rows.append(
            {
                "sample_number": sample_number,
                "text_prompt": text_prompt,
                "best_epoch": best_epoch,
                "semantic_score": semantic_score,
                "geometric_score": geometric_score,
            }
        )

    scores_df = pd.DataFrame(score_rows)
    csv_path = os.path.join(out_dir, "trace_test_scores.csv")
    scores_df.to_csv(csv_path, index=False)
    print(f"Saved trace test scores to: {csv_path}")
    return scores_df

# sample_n = 10
# run_trace_test(result, sample_n) # Run test to see outputs


## Run the experiments

This cell fine-tunes LLaVA‑v1.6 once for each method and then evaluates the trained model on the validation set.

In [ ]:
# Select which methods to run:

# METHODS_TO_RUN = ["bitfit","ia3_adapter", "lora", "qlora"]
# METHODS_TO_RUN = ["bitfit", "ia3_adapter"]
# METHODS_TO_RUN = ["lora", "qlora"]
# METHODS_TO_RUN = ["bitfit"]
# METHODS_TO_RUN = ["ia3_adapter"]
METHODS_TO_RUN = ["lora"]
# METHODS_TO_RUN = ["qlora"]

experiment_outputs = []

for method_name in METHODS_TO_RUN:
    print("=" * 100)
    print(f"Running method: {method_name} -> {METHODS[method_name]['description']}")
    print("=" * 100)

    # Train a LLaVA model using one PEFT method
    result = train_one_method(method_name, train_ds, val_ds)

    # Run test to visualize trace prediction outputs for n validation samples:
    sample_n = 10
    run_trace_test(result, sample_n)

    # Evaluate model performance on validation set.
    metrics, raw_gen_df = evaluate_method_on_val(result["model"], val_samples)
    raw_gen_path = os.path.join(result["run_dir"], "val_generations.csv")
    raw_gen_df.to_csv(raw_gen_path, index=False)

    summary_row = {
        "method": method_name,
        "description": METHODS[method_name]["description"],
        "run_dir": result["run_dir"],
        "train_time_sec": result["train_time_sec"],
        **metrics
    }
    experiment_outputs.append(summary_row)

    # Clean up between methods to reduce GPU memory pressure
    del result
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_outputs).sort_values("trace_score_mean", ascending=False)
results_df


If you need to stop the top block of training early before all methods finish, run this to get the results of the methods you've already completed:

In [ ]:
results_df = pd.DataFrame(experiment_outputs).sort_values("trace_score_mean", ascending=False)
results_df

## Visualize Training Results

In [ ]:

# Save the comparison table
# Store the full method comparison for later analysis.

results_csv = os.path.join(CFG["output_root"], "llava_method_comparison.csv")
results_df.to_csv(results_csv, index=False)
print(results_csv)
results_df


In [ ]:
# Plot method comparison
# Visually compare validation performance across BitFit / IA3 / LoRA / QLoRA.

if len(results_df) > 0:
    plt.figure(figsize=(10, 5))
    plt.bar(results_df["method"], results_df["trace_score_mean"])
    plt.ylabel("Validation Trace Score")
    plt.title("LLaVA-v1.6 Fine-Tuning Method Comparison")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# plot training and val loss curves for each method from the saved history.csv files

for method_name in METHODS_TO_RUN:
    hist_path = os.path.join(CFG["output_root"], method_name, "history.csv")
    if not os.path.exists(hist_path):
        continue

    hist = pd.read_csv(hist_path)
    if hist.empty:
        continue

    plt.figure(figsize=(8, 4))
    if "loss" in hist.columns:
        train_hist = hist[hist["loss"].notna()]
        plt.plot(train_hist["epoch"], train_hist["loss"], label="train")

    best_epoch = None
    best_eval_loss = None
    if "eval_loss" in hist.columns:
        eval_hist = hist[hist["eval_loss"].notna()]
        plt.plot(eval_hist["epoch"], eval_hist["eval_loss"], label="val")

        if not eval_hist.empty:
            best_idx = eval_hist["eval_loss"].idxmin()
            best_epoch = float(eval_hist.loc[best_idx, "epoch"])
            best_eval_loss = float(eval_hist.loc[best_idx, "eval_loss"])
            plt.axvline(
                best_epoch,
                color="red",
                linestyle="--",
                linewidth=1.5,
                label=f"best epoch ({best_epoch:g})",
            )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    title = f"Training curve ({method_name})"
    if best_eval_loss is not None:
        title += f" - best val loss {best_eval_loss:.4f}"
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# TRAINING STOPS HERE

# START INFERENCE ON FINE-TUNED LLaVA MODELS BELOW:

Helper functions for loading model and data:

In [ ]:
# Clear LLaVA objects in memory before loading an inference model.
def clear_llava_gpu_memory(extra_names=None):
    names = {"model", "trainer", "result"}
    if extra_names:
        names.update(extra_names)

    for name in names:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

# Resolve notebook-relative checkpoint paths without treating bad absolute paths as HF repo ids.
def resolve_checkpoint_path(checkpoint_path):
    path = Path(checkpoint_path).expanduser()
    candidates = [path]

    if not path.is_absolute():
        candidates.extend([
            Path.cwd() / path,
            Path.cwd() / "vln-project" / "notebooks" / path,
        ])
    else:
        # Helpful recovery for paths such as /llava_navitrace_runs/lora copied from examples.
        candidates.append(Path.cwd() / str(path).lstrip(os.sep))
        candidates.append(Path.cwd() / "vln-project" / "notebooks" / str(path).lstrip(os.sep))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    tried = "\n".join(f"  - {c}" for c in candidates)
    raise FileNotFoundError(f"Checkpoint directory not found. Tried:\n{tried}")

# Use the run's saved processor when present; otherwise fall back to the base processor.
def load_processor_for_inference(checkpoint_dir, base_model_id):
    processor_source = str(checkpoint_dir) if (checkpoint_dir / "processor_config.json").exists() else base_model_id
    processor = AutoProcessor.from_pretrained(processor_source)
    processor.tokenizer.padding_side = "right"

    if not hasattr(processor, "patch_size") and hasattr(processor, "image_processor"):
        processor.patch_size = getattr(processor.image_processor, "patch_size", None)
    if not hasattr(processor, "vision_feature_select_strategy"):
        processor.vision_feature_select_strategy = "default"

    return processor


# Load a checkpoint and create a fine-tuned llava model
def load_finetuned_llava_model(checkpoint_dir, finetune_method, base_model_id, device, load_adapters_in_4bit=True):
    dtype = torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else (
        torch.float16 if device == "cuda" else torch.float32
    )

    common_kwargs = {
        "torch_dtype": dtype,
        "low_cpu_mem_usage": True,
    }
    if device == "cuda":
        common_kwargs["device_map"] = "auto"

    has_adapter = (checkpoint_dir / "adapter_config.json").exists()

    if not has_adapter:
        model = AutoModelForImageTextToText.from_pretrained(
            str(checkpoint_dir),
            **common_kwargs,
        )
        if device != "cuda":
            model = model.to(device)
        model.eval()
        return model

    if has_adapter and device == "cuda" and (load_adapters_in_4bit or finetune_method == "qlora"):
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
            bnb_4bit_quant_storage=torch.bfloat16 if device == "cuda" else torch.float32,
        )
        common_kwargs["quantization_config"] = quant_config

    base_model = AutoModelForImageTextToText.from_pretrained(
        base_model_id,
        **common_kwargs,
    )
    model = PeftModel.from_pretrained(base_model, str(checkpoint_dir))
    if device != "cuda":
        model = model.to(device)
    model.eval()
    return model

# Find absolute path from local path
def resolve_local_path(path_like):
    path = Path(path_like).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates.extend([
            Path.cwd() / path,
            Path.cwd() / "vln-project" / "notebooks" / path,
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    tried = "\n".join(f"  - {candidate}" for candidate in candidates)
    raise FileNotFoundError(f"Could not find path. Tried:\n{tried}")

# Load a provided local test set and create ground truth traces based on provided annotated csv file.
def load_annotated_test_set(test_set_dir="./test_set", annotated_csv="annotated_paths.csv", image_names=None):
    test_set_dir = resolve_local_path(test_set_dir)
    annotation_path = Path(annotated_csv)
    if not annotation_path.is_absolute():
        annotation_path = test_set_dir / annotation_path
    annotation_path = resolve_local_path(annotation_path)

    annotations = pd.read_csv(annotation_path)
    required_cols = {"image_name", "text_prompt", "point_order", "x", "y"}
    missing_cols = required_cols - set(annotations.columns)
    if missing_cols:
        raise ValueError(f"annotated_paths.csv is missing columns: {sorted(missing_cols)}")

    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = sorted(
        p for p in test_set_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_suffixes
    )
    if image_names is not None:
        wanted = set(image_names)
        image_paths = [p for p in image_paths if p.name in wanted]

    samples = []
    for image_path in image_paths:
        rows = annotations[annotations["image_name"] == image_path.name].copy()
        if rows.empty:
            continue

        rows = rows.sort_values("point_order")
        text_prompt = str(rows["text_prompt"].iloc[0])
        gt_trace_px = rows[["x", "y"]].to_numpy(dtype=np.float32)
        image = Image.open(image_path).convert("RGB")

        samples.append({
            "image_name": image_path.name,
            "image_path": str(image_path),
            "image": image,
            "task": text_prompt,
            "text_prompt": text_prompt,
            "gt_trace_px": gt_trace_px,
        })

    return samples, annotations

# Load fine-tuned llava model based on provided checkpoint map and fine-tune method
def load_finetuned_llava_for_local_inference(
    finetune_method,
    checkpoint_map,
    device=None,
    base_model_id=None,
    cleanup_before_load=True,
    load_adapters_in_4bit=True,
):
    if finetune_method not in checkpoint_map:
        raise ValueError(f"Unknown finetune_method '{finetune_method}'. Available: {list(checkpoint_map.keys())}")
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if base_model_id is None:
        base_model_id = CFG["model_id"]
    if cleanup_before_load:
        clear_llava_gpu_memory()

    checkpoint_dir = resolve_checkpoint_path(checkpoint_map[finetune_method])
    loaded_processor = load_processor_for_inference(checkpoint_dir, base_model_id)
    loaded_model = load_finetuned_llava_model(
        checkpoint_dir,
        finetune_method,
        base_model_id,
        device,
        load_adapters_in_4bit=load_adapters_in_4bit,
    )

    globals()["processor"] = loaded_processor
    return loaded_model, loaded_processor, checkpoint_dir

# def build_inference_messages(image, text_prompt, embodiment=None):
#     """Match the prompt structure used by make_messages_for_sample during fine-tuning."""
#     if embodiment is None:
#         embodiment = CFG.get("embodiment", "Legged Robot") if "CFG" in globals() else "Legged Robot"
#     system_prompt = globals().get(
#         "SYSTEM_PROMPT",
#         (
#             "You are a visual navigation model. Given a scene image and a navigation instruction, "
#             "predict a feasible 2D path as JSON only."
#         ),
#     )
#     user_text = (
#         f"Task: {text_prompt}\n"
#         f"Embodiment: {embodiment}\n"
#         f"Predict the path as JSON only."
#     )

#     return [
#         {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
#         {
#             "role": "user",
#             "content": [
#                 {"type": "image", "image": image},
#                 {"type": "text", "text": user_text},
#             ],
#         },
#     ]


# def pad_image_to_square_for_inference(img):
#     if "pad_image_to_square_for_display" in globals():
#         return pad_image_to_square_for_display(img)

#     w, h = img.size
#     side = max(w, h)
#     pad_left = (side - w) // 2
#     pad_top = (side - h) // 2
#     pad_right = side - w - pad_left
#     pad_bottom = side - h - pad_top
#     return ImageOps.expand(img, border=(pad_left, pad_top, pad_right, pad_bottom), fill=0)


# def extract_first_json_object_for_inference(text):
#     if "extract_first_json_object" in globals():
#         return extract_first_json_object(text)

#     if text is None:
#         return None
#     start = text.find("{")
#     if start < 0:
#         return None

#     depth = 0
#     for i in range(start, len(text)):
#         if text[i] == "{":
#             depth += 1
#         elif text[i] == "}":
#             depth -= 1
#             if depth == 0:
#                 try:
#                     return json.loads(text[start:i + 1])
#                 except Exception:
#                     return None
#     return None


# def parse_generated_trace_for_plot(generated_text, trace_points=None):
#     trace_points = trace_points if trace_points is not None else (CFG.get("trace_points") if "CFG" in globals() else None)
#     payload = extract_first_json_object_for_inference(generated_text)

#     if payload is None:
#         raise ValueError(f"Could not parse a JSON object from model output:\n{generated_text}")

#     if trace_points is not None and "json_to_trace_norm" in globals():
#         trace = json_to_trace_norm(payload, trace_points)
#         if trace is None:
#             raise ValueError(f"Could not parse a valid trace from JSON payload:\n{payload}")
#         return np.asarray(trace, dtype=np.float32), payload

#     raw_trace = payload.get("trace") if isinstance(payload, dict) else None
#     if not isinstance(raw_trace, (list, tuple)):
#         raise ValueError(f"JSON payload does not contain a trace list:\n{payload}")

#     points = []
#     for point in raw_trace:
#         if isinstance(point, dict) and "x" in point and "y" in point:
#             points.append([float(point["x"]), float(point["y"])])
#         elif isinstance(point, (list, tuple)) and len(point) == 2:
#             points.append([float(point[0]), float(point[1])])

#     if not points:
#         raise ValueError(f"Trace list did not contain valid [x, y] points:\n{payload}")

#     return np.clip(np.asarray(points, dtype=np.float32), 0.0, 1.0), payload



# @torch.no_grad()
# def generate_trace_json_for_inference_model(model, processor, sample, max_new_tokens=None):
#     """Generate the raw JSON text for one NaviTrace sample using the loaded inference model."""
#     max_new_tokens = max_new_tokens or (CFG.get("max_new_tokens", 512) if "CFG" in globals() else 512)

#     image = sample["image"].convert("RGB")
#     if "resize_if_needed" in globals() and "CFG" in globals():
#         image = resize_if_needed(image, CFG["image_max_side"])

#     instruction = get_sample_task(sample) if "get_sample_task" in globals() else get_sample_instruction(sample)
#     messages = build_inference_messages(image, instruction)
#     prompt_text = processor.apply_chat_template(
#         messages,
#         tokenize=False,
#         add_generation_prompt=True,
#     )

#     inputs = processor(
#         text=[prompt_text],
#         images=[image],
#         return_tensors="pt",
#         padding=True,
#     )

#     target_device = model.device if hasattr(model, "device") else next(model.parameters()).device
#     inputs = {k: v.to(target_device) if hasattr(v, "to") else v for k, v in inputs.items()}

#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=max_new_tokens,
#         num_beams=CFG.get("generation_num_beams", 1) if "CFG" in globals() else 1,
#         do_sample=False,
#         pad_token_id=processor.tokenizer.eos_token_id,
#     )

#     gen_ids = outputs[:, inputs["input_ids"].shape[1]:]
#     return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()


# def load_test_metadata(metadata_tsv=None):
#     if metadata_tsv is None:
#         candidate = Path("test_predictions_simple.tsv")
#         if not candidate.exists():
#             candidate = Path("vln-project") / "notebooks" / "test_predictions_simple.tsv"
#         metadata_tsv = candidate if candidate.exists() else None

#     if metadata_tsv is None:
#         return None

#     metadata_path = Path(metadata_tsv)
#     candidates = [metadata_path]
#     if not metadata_path.is_absolute():
#         candidates.extend([
#             Path.cwd() / metadata_path,
#             Path.cwd() / "vln-project" / "notebooks" / metadata_path,
#         ])

#     for candidate in candidates:
#         if candidate.exists():
#             return pd.read_csv(candidate, sep="\t", dtype={"sample_id": str})

#     return None


# def nonempty_value(*values, default=""):
#     for value in values:
#         if value is None:
#             continue
#         if isinstance(value, float) and pd.isna(value):
#             continue
#         if isinstance(value, str) and value.strip() == "":
#             continue
#         return value
#     return default


# def format_category_for_tsv(category):
#     if isinstance(category, (list, tuple)):
#         return json.dumps(list(category))
#     return str(category)


# def sample_metadata_for_export(sample, index, metadata_df=None, embodiment=None):
#     metadata_row = metadata_df.iloc[index] if metadata_df is not None and index < len(metadata_df) else {}
#     row_get = metadata_row.get if hasattr(metadata_row, "get") else (lambda key, default=None: default)
#     default_embodiment = embodiment or (CFG.get("embodiment", "Legged Robot") if "CFG" in globals() else "Legged Robot")

#     sample_id = nonempty_value(
#         sample.get("sample_id"),
#         sample.get("id"),
#         sample.get("uid"),
#         row_get("sample_id", ""),
#     )
#     category = nonempty_value(
#         sample.get("category"),
#         sample.get("scene_category"),
#         row_get("category", ""),
#         default="unknown",
#     )
#     embodiment = nonempty_value(
#         sample.get("embodiment"),
#         row_get("embodiment", ""),
#         default_embodiment,
#     )

#     return {
#         "sample_id": str(sample_id),
#         "embodiment": str(embodiment),
#         "category": format_category_for_tsv(category),
#     }



# def remove_padding_and_renormalize_trace(trace_norm, sample):
#     """Convert padded-square normalized coordinates into original-image normalized coordinates."""
#     trace_norm = np.asarray(trace_norm, dtype=np.float32)
#     orig_w, orig_h = sample["image"].size
#     side = max(orig_w, orig_h)
#     pad_left = (side - orig_w) / 2.0
#     pad_top = (side - orig_h) / 2.0

#     trace_orig_norm = trace_norm.copy()
#     trace_orig_norm[:, 0] = (trace_orig_norm[:, 0] * side - pad_left) / orig_w
#     trace_orig_norm[:, 1] = (trace_orig_norm[:, 1] * side - pad_top) / orig_h
#     return np.clip(trace_orig_norm, 0.0, 1.0)


# def format_prediction_for_tsv(trace_norm, sample, prediction_format="pixel"):
#     trace_orig_norm = remove_padding_and_renormalize_trace(trace_norm, sample)

#     if prediction_format == "normalized":
#         return trace_orig_norm.tolist()
#     if prediction_format != "pixel":
#         raise ValueError("prediction_format must be 'pixel' or 'normalized'")

#     orig_w, orig_h = sample["image"].size
#     px = trace_orig_norm.copy()
#     px[:, 0] = px[:, 0] * orig_w
#     px[:, 1] = px[:, 1] * orig_h
#     px[:, 0] = np.clip(px[:, 0], 0, orig_w - 1)
#     px[:, 1] = np.clip(px[:, 1], 0, orig_h - 1)
#     return px.tolist()

# def fallback_trace(trace_points=None):
#     trace_points = trace_points or (CFG.get("trace_points", 9) if "CFG" in globals() else 9)
#     return np.tile(np.array([[0.5, 0.5]], dtype=np.float32), (trace_points, 1)).tolist()


# def export_finetuned_method_test_predictions(
#     finetune_method,
#     checkpoint_map,
#     samples=None,
#     out_tsv=None,
#     metadata_tsv=None,
#     max_new_tokens=None,
#     prediction_format="pixel",
#     device=None,
#     base_model_id=None,
#     cleanup_before_load=True,
#     load_adapters_in_4bit=True,
# ):
#     """
#     Load one fine-tuned LLaVA method, then export predictions with export_test_predictions().

#     This intentionally uses the same generation, parsing, fallback, metadata, and coordinate
#     formatting path as the experiment-running code block. The only extra work here is loading
#     the requested saved checkpoint before delegating to export_test_predictions().
#     """
#     if out_tsv is None:
#         output_root = CFG.get("output_root", "./llava_navitrace_runs") if "CFG" in globals() else "./llava_navitrace_runs"
#         out_tsv = os.path.join(output_root, finetune_method, f"test_predictions_{finetune_method}.tsv")

#     if device is None:
#         device = "cuda" if torch.cuda.is_available() else "cpu"
#     if base_model_id is None:
#         base_model_id = CFG["model_id"] if "CFG" in globals() else "llava-hf/llava-v1.6-mistral-7b-hf"
#     if max_new_tokens is None:
#         max_new_tokens = CFG.get("max_new_tokens", 512) if "CFG" in globals() else 512

#     if cleanup_before_load:
#         clear_llava_gpu_memory()

#     checkpoint_dir = resolve_checkpoint_path(checkpoint_map[finetune_method])
#     processor = load_processor_for_inference(checkpoint_dir, base_model_id)
#     model = load_finetuned_llava_model(
#         checkpoint_dir,
#         finetune_method,
#         base_model_id,
#         device,
#         load_adapters_in_4bit=load_adapters_in_4bit,
#     )

#     # export_test_predictions() calls generate_trace_json_test(), which uses the global processor.
#     # Point that global at the processor saved with the checkpoint so this path matches training export.
#     globals()["processor"] = processor

#     out_path = Path(out_tsv)
#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     export_test_predictions(
#         model,
#         samples,
#         str(out_path),
#         metadata_tsv=metadata_tsv,
#         prediction_format=prediction_format,
#     )
#     return pd.read_csv(out_path, sep="\t", dtype={"sample_id": str})

# def plot_llava_inference_overlay(image, trace_norm, title=None):
#     img_sq = pad_image_to_square_for_inference(image.convert("RGB"))
#     trace = np.asarray(trace_norm, dtype=np.float32)
#     side = img_sq.size[0]
#     trace_px = trace * side

#     fig, ax = plt.subplots(figsize=(8, 8))
#     ax.imshow(img_sq)
#     ax.plot(trace_px[:, 0], trace_px[:, 1], marker="o", linewidth=2.5, label="Predicted path")
#     ax.scatter(trace_px[0, 0], trace_px[0, 1], s=90, marker="s", label="Start")
#     ax.scatter(trace_px[-1, 0], trace_px[-1, 1], s=120, marker="*", label="Goal")
#     ax.set_axis_off()
#     ax.legend(loc="upper right")
#     if title:
#         ax.set_title(title)
#     fig.tight_layout(pad=0)
#     plt.show()
#     return fig, ax


# def run_llava_inference(
#     image,
#     text_prompt,
#     finetune_method,
#     checkpoint_map,
#     max_new_tokens=None,
#     device=None,
#     base_model_id=None,
#     return_model=False,
#     show_plot=True,
#     return_trace=False,
# ):
#     """
#     Run inference with the trained weights saved by this notebook.

#     LoRA, QLoRA, and IA3 checkpoints are saved as PEFT adapters, so this loads the
#     base LLaVA model first and then attaches the adapter weights. BitFit is saved
#     as a full model checkpoint and is loaded directly.
#     """
#     if finetune_method not in checkpoint_map:
#         raise ValueError(
#             f"Unknown finetune_method '{finetune_method}'. "
#             f"Available: {list(checkpoint_map.keys())}"
#         )

#     if device is None:
#         device = "cuda" if torch.cuda.is_available() else "cpu"
#     if base_model_id is None:
#         base_model_id = CFG["model_id"] if "CFG" in globals() else "llava-hf/llava-v1.6-mistral-7b-hf"
#     if max_new_tokens is None:
#         max_new_tokens = CFG.get("max_new_tokens", 512) if "CFG" in globals() else 512

#     checkpoint_dir = resolve_checkpoint_path(checkpoint_map[finetune_method])

#     if isinstance(image, (str, os.PathLike)):
#         image = Image.open(image).convert("RGB")
#     elif isinstance(image, Image.Image):
#         image = image.convert("RGB")
#     else:
#         raise TypeError("image must be a PIL image or a file path string")

#     if "resize_if_needed" in globals() and "CFG" in globals():
#         image = resize_if_needed(image, CFG["image_max_side"])

#     processor = load_processor_for_inference(checkpoint_dir, base_model_id)
#     model = load_finetuned_llava_model(checkpoint_dir, finetune_method, base_model_id, device)

#     # Match export_test_predictions(): generate_trace_json_test() reads the global processor.
#     globals()["processor"] = processor
#     sample_for_inference = {
#         "image": image,
#         "task": text_prompt,
#     }
#     generated_text = generate_trace_json_test(model, sample_for_inference).strip()

#     trace_norm = None
#     payload = None
#     if show_plot or return_trace:
#         trace_norm, payload = parse_generated_trace_for_plot(generated_text)

#     if show_plot:
#         plot_llava_inference_overlay(
#             image,
#             trace_norm,
#             title=f"{finetune_method} prediction",
#         )

#     outputs = {"text": generated_text, "trace": trace_norm, "payload": payload}
#     if return_model:
#         outputs.update({"model": model, "processor": processor})
#     if return_trace or return_model:
#         return outputs
#     return generated_text

# # Produce leaderboard-style TSV output for the test split.
# def export_test_predictions(model, samples, out_tsv, metadata_tsv="./test_predictions_simple.tsv", prediction_format="pixel"):
#     if prediction_format not in ("pixel", "normalized"):
#         raise ValueError("prediction_format must be 'pixel' or 'normalized'")

#     metadata_df = load_test_metadata(metadata_tsv)
#     rows = []

#     for i, sample in enumerate(tqdm(samples, desc="Generating test predictions")):
#         meta = sample_metadata_for_export(sample, i, metadata_df)

#         try:
#             gen_text = generate_trace_json_test(model, sample)
#             payload = extract_first_json_object(gen_text)
#             pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
#         except Exception as e:
#             print(f"Warning: failed on sample {sample.get('sample_id', 'unknown')}: {e}")
#             pred_norm = [[0.5, 0.5] for _ in range(CFG["trace_points"])]

#         if pred_norm is None:
#             pred_norm = [[0.5, 0.5] for _ in range(CFG["trace_points"])]

#         orig_w, orig_h = sample["image"].size
#         pred_px = pred_padded_to_pixel(pred_norm, orig_w, orig_h)
#         if prediction_format == "pixel":
#             prediction = pred_px.tolist()
#         else:
#             pred_orig_norm = pred_px.copy()
#             pred_orig_norm[:, 0] = pred_orig_norm[:, 0] / orig_w
#             pred_orig_norm[:, 1] = pred_orig_norm[:, 1] / orig_h
#             prediction = np.clip(pred_orig_norm, 0.0, 1.0).tolist()

#         rows.append({
#             **meta,
#             "prediction": json.dumps(prediction),
#         })

#     pd.DataFrame(rows, columns=["sample_id", "embodiment", "category", "prediction"]).to_csv(out_tsv, sep="\t", index=False)
#     print(f"Saved test predictions to: {out_tsv}")


Helper functions for making inferences with different sample formats

In [ ]:
def original_trace_to_padded_square_px(trace_px, image):
    trace_px = np.asarray(trace_px, dtype=np.float32).copy()
    orig_w, orig_h = image.size
    side = max(orig_w, orig_h)
    pad_left = (side - orig_w) / 2.0
    pad_top = (side - orig_h) / 2.0
    trace_px[:, 0] = trace_px[:, 0] + pad_left
    trace_px[:, 1] = trace_px[:, 1] + pad_top
    return trace_px


def model_norm_trace_to_padded_square_px(trace_norm, image):
    trace_norm = np.asarray(trace_norm, dtype=np.float32)
    side = max(image.size)
    pred_px = trace_norm.copy()
    pred_px[:, 0] = pred_px[:, 0] * side
    pred_px[:, 1] = pred_px[:, 1] * side
    pred_px[:, 0] = np.clip(pred_px[:, 0], 0, side - 1)
    pred_px[:, 1] = np.clip(pred_px[:, 1], 0, side - 1)
    return pred_px

def model_norm_trace_to_regular_img_size(trace_norm, image):
    trace_norm = np.asarray(trace_norm, dtype=np.float32)
    w, h = image.size

    pred_px = trace_norm.copy()
    pred_px[:, 0] = pred_px[:, 0] * w
    pred_px[:, 1] = pred_px[:, 1] * h

    pred_px[:, 0] = np.clip(pred_px[:, 0], 0, w - 1)
    pred_px[:, 1] = np.clip(pred_px[:, 1], 0, h - 1)
    return pred_px

def model_padded_square_norm_trace_to_image_px(trace_norm, image):
    trace_norm = np.asarray(trace_norm, dtype=np.float32)
    w, h = image.size
    side = max(w, h)

    pad_left = (side - w) / 2.0
    pad_top = (side - h) / 2.0

    pred_px = trace_norm.copy()
    pred_px[:, 0] = pred_px[:, 0] * side - pad_left
    pred_px[:, 1] = pred_px[:, 1] * side - pad_top

    pred_px[:, 0] = np.clip(pred_px[:, 0], 0, w - 1)
    pred_px[:, 1] = np.clip(pred_px[:, 1], 0, h - 1)
    return pred_px

def predict_test_trace_normalized_vals(model, sample):
    # Use the same generation path as validation: generate_trace_json_val -> make_messages_for_sample.
    gen_text = generate_trace_json_val(model, sample)
    payload = extract_first_json_object(gen_text)
    pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
    if pred_norm is None:
        return None, gen_text, payload
    
    pred_norm = np.asarray(pred_norm, dtype=np.float32)
    
    return pred_norm, gen_text, payload

def predict_test_trace_px(model, sample):
    # Use the same generation path as validation: generate_trace_json_val -> make_messages_for_sample.
    gen_text = generate_trace_json_val(model, sample)
    payload = extract_first_json_object(gen_text)
    pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
    if pred_norm is None:
        return None, gen_text, payload

    # pred_px = model_norm_trace_to_padded_square_px(pred_norm, sample["image"])
    pred_px = model_padded_square_norm_trace_to_image_px(pred_norm, sample["image"])
    return pred_px, gen_text, payload

def predict_padded_test_trace_px(model, sample):
    # Use the same generation path as validation: generate_trace_json_val -> make_messages_for_sample.
    gen_text = generate_trace_json_val(model, sample)
    payload = extract_first_json_object(gen_text)
    pred_norm = json_to_trace_norm(payload, CFG["trace_points"])
    if pred_norm is None:
        return None, gen_text, payload

    pred_px = model_norm_trace_to_padded_square_px(pred_norm, sample["image"])
    return pred_px, gen_text, payload

def plot_padded_test_trace_overlay(output_dir, plot, save_plot, sample, image_name, pred_trace_px=None, title_extra=None):
    image = sample["image"]
    padded_image, _ = pad_to_square(image)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(padded_image)

    if "gt_trace_px" in sample:
        gt_trace_px = original_trace_to_padded_square_px(sample["gt_trace_px"], image)

        if len(gt_trace_px) > 0:
            ax.plot(gt_trace_px[:, 0], gt_trace_px[:, 1], marker="o", linewidth=2.5, color="lime", label="Annotated path")
            ax.scatter(gt_trace_px[0, 0], gt_trace_px[0, 1], s=90, marker="s", color="lime", edgecolor="black", label="Annotated start")
            ax.scatter(gt_trace_px[-1, 0], gt_trace_px[-1, 1], s=130, marker="*", color="lime", edgecolor="black", label="Annotated goal")

    elif "ground_truth" in sample:
        # print(sample["ground_truth"])
        # gt_trace_px = original_trace_to_padded_square_px(sample["ground_truth"], image)
        gt_traces = sample["ground_truth"].get(CFG["embodiment"], [])
        if len(gt_traces) > 0:
            gt_trace_px = original_trace_to_padded_square_px(gt_traces[0], image)

            if len(gt_trace_px) > 0:
                ax.plot(gt_trace_px[:, 0], gt_trace_px[:, 1], marker="o", linewidth=2.5, color="lime", label="Annotated path")
                ax.scatter(gt_trace_px[0, 0], gt_trace_px[0, 1], s=90, marker="s", color="lime", edgecolor="black", label="Annotated start")
                ax.scatter(gt_trace_px[-1, 0], gt_trace_px[-1, 1], s=130, marker="*", color="lime", edgecolor="black", label="Annotated goal")

    if pred_trace_px is not None:
        pred_trace_px = np.asarray(pred_trace_px, dtype=np.float32)
        ax.plot(pred_trace_px[:, 0], pred_trace_px[:, 1], marker="o", linewidth=2.5, color="red", label="Predicted path")
        ax.scatter(pred_trace_px[0, 0], pred_trace_px[0, 1], s=90, marker="s", color="red", edgecolor="black", label="Predicted start")
        ax.scatter(pred_trace_px[-1, 0], pred_trace_px[-1, 1], s=130, marker="*", color="red", edgecolor="black", label="Predicted goal")

    title = image_name + f" | {sample['task']}"
    
    if title_extra:
        title += f" | {title_extra}"

    ax.set_title(title, fontsize=10)
    side = padded_image.size[0]
    ax.set_xlim(0, side)
    ax.set_ylim(side, 0)
    ax.set_axis_off()
    ax.legend(loc="upper right")
    fig.tight_layout()

    if save_plot:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        plt.savefig(output_dir + image_name)  # Saves the plot to the current directory

    if plot:
        plt.show()

    return fig, ax

def plot_test_trace_overlay(output_dir, plot, save_plot, sample, image_name, pred_trace_px=None, title_extra=None):
    image = sample["image"]
    # padded_image, _ = pad_to_square(image)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(image)

    if "gt_trace_px" in sample:
        # gt_trace_px = original_trace_to_padded_square_px(sample["gt_trace_px"], image)
        gt_trace_px = np.asarray(sample["gt_trace_px"], dtype=np.float32)

        if len(gt_trace_px) > 0:
            ax.plot(gt_trace_px[:, 0], gt_trace_px[:, 1], marker="o", linewidth=2.5, color="lime", label="Annotated path")
            ax.scatter(gt_trace_px[0, 0], gt_trace_px[0, 1], s=90, marker="s", color="lime", edgecolor="black", label="Annotated start")
            ax.scatter(gt_trace_px[-1, 0], gt_trace_px[-1, 1], s=130, marker="*", color="lime", edgecolor="black", label="Annotated goal")

    if pred_trace_px is not None:
        pred_trace_px = np.asarray(pred_trace_px, dtype=np.float32)
        ax.plot(pred_trace_px[:, 0], pred_trace_px[:, 1], marker="o", linewidth=2.5, color="red", label="Predicted path")
        ax.scatter(pred_trace_px[0, 0], pred_trace_px[0, 1], s=90, marker="s", color="red", edgecolor="black", label="Predicted start")
        ax.scatter(pred_trace_px[-1, 0], pred_trace_px[-1, 1], s=130, marker="*", color="red", edgecolor="black", label="Predicted goal")

    title = image_name + f" | {sample['task']}"
    
    if title_extra:
        title += f" | {title_extra}"

    ax.set_title(title, fontsize=10)
    # side = padded_image.size[0]
    # ax.set_xlim(0, side)
    # ax.set_ylim(side, 0)
    w, h = image.size
    ax.set_xlim(0, w)
    ax.set_ylim(h, 0)
    ax.set_axis_off()
    ax.legend(loc="upper right")
    fig.tight_layout()

    if save_plot:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        plt.savefig(output_dir + image_name)  # Saves the plot to the current directory

    if plot:
        plt.show()

    return fig, ax


def run_test_set_inference(
    finetune_method,
    checkpoint_map,
    samples,
    out_plot_dir,
    max_image_num=None,
    use_padded=False,
    plot_trace_plots=False,
    save_trace_plots=False,
    load_adapters_in_4bit=True,
):
    model, loaded_processor, checkpoint_dir = load_finetuned_llava_for_local_inference(
        finetune_method,
        checkpoint_map,
        load_adapters_in_4bit=load_adapters_in_4bit,
    )

    if max_image_num is not None:
        samples = samples[:max_image_num]

    results = []
    for i, sample in enumerate(tqdm(samples, desc=f"Displaying {finetune_method} local test traces")):
        if use_padded:
            pred_trace_px, gen_text, payload = predict_padded_test_trace_px(model, sample)
        else:
            pred_trace_px, gen_text, payload = predict_test_trace_px(model, sample)

        title_extra = "parsed" if pred_trace_px is not None else "parse failed"

        img_name = f"sample_num = {i}"
        if "image_name" in sample:
            img_name = sample["image_name"]
        
        if use_padded:
            plot_padded_test_trace_overlay(out_plot_dir, plot_trace_plots, save_trace_plots, sample, img_name, pred_trace_px=pred_trace_px, title_extra=title_extra)
        else:
            plot_test_trace_overlay(out_plot_dir, plot_trace_plots, save_trace_plots, sample, img_name, pred_trace_px=pred_trace_px, title_extra=title_extra)

        results.append({
            "image_name": img_name,
            "task": sample["task"],
            "text_prompt": sample["task"],
            "pred_trace_px": None if pred_trace_px is None else pred_trace_px.tolist(),
            "raw_generation": gen_text,
            "checkpoint_dir": str(checkpoint_dir),
        })

        if "gt_trace_px" in sample:
            results.append({"gt_trace_px_padded_square": original_trace_to_padded_square_px(sample["gt_trace_px"], sample["image"]).tolist()})

        # print(results)

    return pd.DataFrame(results)

def export_test_predictions(
    finetune_method,
    checkpoint_map,
    samples,
    plot_trace,
    out_plot_dir,
    plot_num,
    use_normalized = False,
    use_padded = False,
    out_tsv=None,
    metadata_tsv=None,
    max_new_tokens=None,
    prediction_format="pixel",
    device=None,
    base_model_id=None,
    cleanup_before_load=True,
    load_adapters_in_4bit=True,
):
    if out_tsv is None:
        output_root = CFG.get("output_root", "./llava_navitrace_runs") if "CFG" in globals() else "./llava_navitrace_runs"
        out_tsv = os.path.join(output_root, finetune_method, f"test_predictions_{finetune_method}.tsv")

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if base_model_id is None:
        base_model_id = CFG["model_id"] if "CFG" in globals() else "llava-hf/llava-v1.6-mistral-7b-hf"
    if max_new_tokens is None:
        max_new_tokens = CFG.get("max_new_tokens", 512) if "CFG" in globals() else 512

    if cleanup_before_load:
        clear_llava_gpu_memory()

    # checkpoint_dir = _resolve_checkpoint_path(checkpoint_map[finetune_method])
    # processor = _load_processor_for_inference(checkpoint_dir, base_model_id)
    # model = _load_finetuned_llava_model(
    #     checkpoint_dir,
    #     finetune_method,
    #     base_model_id,
    #     device,
    #     load_adapters_in_4bit=load_adapters_in_4bit,
    # )

    model, loaded_processor, checkpoint_dir = load_finetuned_llava_for_local_inference(
        finetune_method,
        checkpoint_map,
        load_adapters_in_4bit=load_adapters_in_4bit,
    )

    # globals()["processor"] = processor

    out_path = Path(out_tsv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    export_test_predictions_helper(
        model,
        samples,
        str(out_path),
        out_plot_dir,
        use_normalized=use_normalized,
        use_padded=use_padded,
        plot_trace=plot_trace,
        plot_num=plot_num,
        metadata_tsv=metadata_tsv,
        prediction_format=prediction_format,
    )
    return pd.read_csv(out_path, sep="\t", dtype={"sample_id": str})

def export_test_predictions_helper(model, samples, out_tsv, out_plot_dir, use_normalized=False, use_padded=False, plot_trace=True, plot_num=10, metadata_tsv="./test_predictions_simple.tsv", prediction_format="pixel"):

    results = []
    for i, sample in enumerate(tqdm(samples, desc="Generating test predictions")):

        if use_normalized:
            pred_trace_px, gen_text, payload = predict_test_trace_normalized_vals(model, sample)
        else:
            if use_padded:
                pred_trace_px, gen_text, payload = predict_padded_test_trace_px(model, sample)
            else:
                pred_trace_px, gen_text, payload = predict_test_trace_px(model, sample)

            orig_w, orig_h = sample["image"].size

            img_name = f"sample_num = {i}"
            # if "image_name" in sample:
            #     img_name = sample["image_name"]

            if plot_trace and i < plot_num:
                if use_padded:
                    plot_padded_test_trace_overlay(out_plot_dir, True, True, sample, img_name, pred_trace_px=pred_trace_px, title_extra="")
                else:
                    plot_test_trace_overlay(out_plot_dir, True, True, sample, img_name, pred_trace_px=pred_trace_px, title_extra="")

        if pred_trace_px is not None:
            prediction = pred_trace_px.tolist()
        else:
            prediction = None

        # Save results to dict
        results.append({
        "sample_id":  sample["sample_id"],
        "embodiment": CFG["embodiment"],
        "category":  sample["category"],
        "prediction": prediction,
        })

    # Save all results to tsv file
    pd.DataFrame(results).to_csv(out_tsv, sep="\t", index=False)
    print(f"Saved test predictions to: {out_tsv}")



Make checkpoint map to create model for inferences:

Directions: 
- Map the checkpoint directory path the the PEFT method you are using for inferences

In [ ]:
checkpoint_map = {
    "bitfit": "./llava_navitrace_runs/bitfit",
    "ia3_adapter": "./llava_navitrace_runs/ia3_adapter",
    "lora": "./llava_navitrace_runs/lora/checkpoint-663",
    "qlora": "./llava_navitrace_runs/qlora/checkpoint-612",
}

finetune_method = "qlora"

load_4bit_adapters = False
if finetune_method == "qlora":
    load_4bit_adapters = True

## Inference on NaviTrace validation set

In [ ]:
val_num = 10
val_samples = val_samples[:val_num]

print(val_samples[0].keys())

out_plot_dir = "./llava_navitrace_runs/" + finetune_method + "/val_output_plots/"

plot_num = val_num

val_predictions_df = export_test_predictions(
    finetune_method=finetune_method,
    checkpoint_map=checkpoint_map,
    samples=val_samples,
    use_padded=True,
    plot_trace=True,
    out_plot_dir=out_plot_dir,
    plot_num=10,
    out_tsv=None,
    metadata_tsv=None,
    max_new_tokens=None,
    prediction_format="pixel",
    device=None,
    base_model_id=None,
    cleanup_before_load=True,
    load_adapters_in_4bit=load_4bit_adapters,
)

val_predictions_df.head()

## Inference on in-home test set:

In [ ]:
# Load test set
test_set_dir = "./test_set_2" # Map this to annotated test set
annotated_csv = "annotated_paths.csv"
image_names = None
out_plot_dir = test_set_dir + "/plots/"
samples, annotations = load_annotated_test_set(
    test_set_dir=test_set_dir,
    annotated_csv=annotated_csv,
    image_names=image_names,
)

local_overlay_results = run_test_set_inference(
    finetune_method=finetune_method,
    checkpoint_map=checkpoint_map,
    samples=samples,
    out_plot_dir= test_set_dir + "/" + finetune_method + "_inferences/",
    use_padded=True,
    max_image_num=None,
    plot_trace_plots=True,
    save_trace_plots=True,
    load_adapters_in_4bit=load_4bit_adapters,
)

local_overlay_results[["image_name", "text_prompt", "pred_trace_px"]]


## Inference on NaviTrace Test set

In [ ]:
# Load the test split for later leaderboard-style export
dataset = load_dataset("leggedrobotics/navitrace")

test_samples = []
for s in tqdm(list(dataset["test"]), desc="Filtering test"):
    if CFG["embodiment"] in s.get("embodiments", []):
        test_samples.append(s)

In [ ]:
# Get small sub-set of test set
num_test_samples = None
test_samples_copy = test_samples.copy()
if num_test_samples is not None:
    test_samples_copy = test_samples[:num_test_samples]

In [ ]:
out_plot_dir = "./llava_navitrace_runs/" + finetune_method + "/test_output_plots/"

plot_num = 10
if num_test_samples is not None:
    plot_num = min(10, num_test_samples)

test_predictions_df = export_test_predictions(
    finetune_method=finetune_method,
    checkpoint_map=checkpoint_map,
    samples=test_samples_copy,
    plot_trace=True,
    out_plot_dir=out_plot_dir,
    plot_num=10,
    use_normalized=True,
    use_padded=False,
    out_tsv=None,
    metadata_tsv=None,
    max_new_tokens=None,
    prediction_format="pixel",
    device=None,
    base_model_id=None,
    cleanup_before_load=True,
    load_adapters_in_4bit=load_4bit_adapters,
)

test_predictions_df.head()
